<a href="https://colab.research.google.com/github/festusoladayoonline-debug/Beyond-Potency-NorA/blob/main/BmrA_tpr_system_composition_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BmrA system composition & motif mapping from a GROMACS `.tpr` (Google Colab)

Upload `.tpr` and this notebook will report, directly from the topology and its starting coordinates:

1. System composition (segments, residue ranges, atom counts)
2. Walker A and signature-C (LSGGQ) motifs in each monomer, found by sequence scan
3. Which ATP / Mg²⁺ sits at which Walker A, and the composite signature-C partner (from starting coordinates)
4. Q-loop candidate Gln residues (ranked by distance to the site's Mg²⁺)
5. Glu326 salt-bridge partner check

Nothing is hard-coded to residue numbers: everything is derived from the file, so it also works for other BmrA constructs.

## 1. Install

In [ ]:
!pip -q install MDAnalysis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.3 MB/s eta 0:00:00


## 2. Upload your `.tpr`
A file picker will open. Choose your topology file (e.g. `topol.tpr` or `ext_to_1_microS.tpr`).

In [ ]:
from google.colab import files
uploaded = files.upload()
TPR_PATH = list(uploaded.keys())[0]
print("Using:", TPR_PATH)

Saving topol (1).tpr to topol (1).tpr
Using: topol (1).tpr


## 3. Analysis functions

In [ ]:
import re, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import distances as mdd

AA = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','HSD':'H','HSE':'H',
      'HSP':'H','HID':'H','HIE':'H','HIP':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
      'THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
WALKER_A_REGEX = r"G..G.GK[ST]"
SIG_C_MOTIF = "LSGGQ"
WATER = {'TIP3','SOL','WAT','HOH','TIP4','SPC'}

def coord_scale(u):
    """A bare .tpr can hand back coordinates in nm (no trajectory reader to convert units).
    Detect this from typical bond lengths (~1-1.5 A, i.e. ~0.1-0.15 nm) and return the factor to Angstrom."""
    idx = u.bonds.indices[:2000]
    d = np.linalg.norm(u.atoms.positions[idx[:, 0]] - u.atoms.positions[idx[:, 1]], axis=1)
    return 10.0 if np.median(d) < 0.5 else 1.0

def analyse(tpr, glu_resid=326):
    u = mda.Universe(tpr)
    S = coord_scale(u)
    pos = lambda ag: ag.positions * S
    out = {}
    # 1. composition
    rows = []
    for seg in u.segments:
        r = seg.residues
        rows.append(dict(segid=seg.segid, resnames=("protein (%d residue types)" % len(set(r.resnames))) if len(seg.atoms.select_atoms("protein"))==len(seg.atoms) else ",".join(sorted(set(r.resnames))), n_residues=len(r),
                         resid_min=int(r.resids.min()), resid_max=int(r.resids.max()), n_atoms=len(seg.atoms),
                         atoms_per_residue=(len(seg.atoms)//len(r)) if len(set(len(x.atoms) for x in r[:50]))==1 else "varies"))
    out['composition'] = pd.DataFrame(rows)
    # 2. protein chains + motifs
    chains = {}
    for seg in u.segments:
        prot = seg.atoms.select_atoms("protein")
        if len(prot) == 0: continue
        res = prot.residues
        seq = "".join(AA.get(n,'X') for n in res.resnames)
        chains[seg.segid] = dict(res=res, resids=res.resids, seq=seq)
    out['chains'] = chains
    motifs, mrows = {}, []
    for sid, c in chains.items():
        m = re.search(WALKER_A_REGEX, c['seq']); s = c['seq'].find(SIG_C_MOTIF)
        if not m or s < 0: continue
        wa = (int(c['resids'][m.start()]), int(c['resids'][m.end()-1]))
        lys = int(c['resids'][m.start()+6]); ser = int(c['resids'][s+1])
        sc = (int(c['resids'][s]), int(c['resids'][s+4]))
        motifs[sid] = dict(walkerA=wa, walkerA_lys=lys, walkerA_seq=m.group(), sigC=sc, sigC_ser=ser)
        mrows.append(dict(segid=sid, walkerA_motif=m.group(), walkerA_resids=f"{wa[0]}-{wa[1]}", walkerA_Lys=lys,
                          sigC_resids=f"{sc[0]}-{sc[1]}", sigC_Ser=ser))
    out['motifs'] = motifs; out['motif_table'] = pd.DataFrame(mrows)
    # 3. ligands
    atp = u.select_atoms("resname ATP"); mg = u.select_atoms("resname MG MG2 MGA")
    out['atp_resids'] = list(atp.residues.resids); out['mg_resids'] = list(mg.residues.resids)
    out['atp_atom_names'] = list(atp.residues[0].atoms.names) if len(atp) else []
    # 4. pairing: each ATP -> nearest Walker A Lys NZ (site), each Mg -> nearest ATP phosphate
    pair = []
    for r in atp.residues:
        pg = pos(r.atoms.select_atoms("name PG"))[0]
        best = None
        for sid, m in motifs.items():
            nz = u.select_atoms(f"segid {sid} and resid {m['walkerA_lys']} and name NZ")
            d = np.linalg.norm(pos(nz)[0]-pg)
            if best is None or d < best[1]: best = (sid, d, m)
        sid, d, m = best
        partner = [k for k in motifs if k != sid][0]
        # Mg
        phos = pos(r.atoms.select_atoms("name PA PB PG"))
        mgd = [(x.resid, float(np.min(np.linalg.norm(phos - pos(x.atoms)[0], axis=1)))) for x in mg.residues]
        mgres, mgdist = min(mgd, key=lambda t: t[1])
        # signature-C distance of partner chain
        sc_ca = u.select_atoms(f"segid {partner} and resid {motifs[partner]['sigC_ser']} and name CA")
        wa_ca = u.select_atoms(f"segid {sid} and resid {m['walkerA_lys']} and name CA")
        pair.append(dict(ATP_resid=int(r.resid), bound_at_WalkerA_of=sid, WalkerA_Lys=m['walkerA_lys'],
                         Lys_NZ_to_PG_A=round(d,2), composite_SigC_partner=partner, SigC_Ser=motifs[partner]['sigC_ser'],
                         paired_Mg_resid=int(mgres), Mg_to_ATP_phosphate_A=round(mgdist,2),
                         closure_CA_CA_start_A=round(float(np.linalg.norm(pos(wa_ca)[0]-pos(sc_ca)[0])),2)))
    out['pairing'] = pd.DataFrame(pair)
    # 5. Q-loop candidates: Gln between Walker A and signature C in each chain, ranked by distance to that site's Mg
    q = []
    for _, p in out['pairing'].iterrows():
        sid = p.bound_at_WalkerA_of; m = motifs[sid]
        mgpos = pos(u.select_atoms(f"resid {p.paired_Mg_resid}"))[0]
        for r in chains[sid]['res']:
            if r.resname == 'GLN' and m['walkerA'][1] < r.resid < m['sigC'][0]:
                sc = u.select_atoms(f"segid {sid} and resid {r.resid} and name NE2 OE1")
                d = np.min(np.linalg.norm(pos(sc)-mgpos, axis=1))
                q.append(dict(segid=sid, Gln_resid=int(r.resid), dist_to_site_Mg_A=round(float(d),2)))
    out['qloop'] = pd.DataFrame(q).sort_values(['segid','dist_to_site_Mg_A'])
    # 6. salt bridge check for Glu
    sb = []
    for sid, c in chains.items():
        rid = int(c['resids'][0]) + glu_resid - 1      # same residue number within each monomer
        g = u.select_atoms(f"segid {sid} and resid {rid} and name OE1 OE2")
        if len(g)==0 or g.resnames[0] != 'GLU': continue
        lys = u.select_atoms("resname LYS and name NZ")
        d = mdd.distance_array(pos(g), pos(lys)).min(axis=0)
        for i in np.argsort(d)[:5]:
            sb.append(dict(Glu_segid=sid, Glu_resid=rid, Lys_segid=lys[i].segid, Lys_resid=int(lys[i].resid), dist_A=round(float(d[i]),2)))
    out['saltbridge'] = pd.DataFrame(sb)
    out['scale'] = S
    out['n_atoms'] = len(u.atoms)
    return u, out


## 4. Run

In [ ]:
GLU_RESID = 326   # residue to test for a salt-bridge partner (numbering within each monomer)
u, res = analyse(TPR_PATH, glu_resid=GLU_RESID)
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(f"Atoms: {res['n_atoms']:,} | coordinate scale factor applied to reach Angstrom: x{res['scale']:.0f} "
      f"({'tpr coordinates were in nm' if res['scale']==10 else 'already Angstrom'})")

Atoms: 188,396 | coordinate scale factor applied to reach Angstrom: x10 (tpr coordinates were in nm)


### 4a. System composition

In [ ]:
display(res["composition"])

,segid,resnames,n_residues,resid_min,resid_max,n_atoms,atoms_per_residue
0,seg_0_PROA,protein (20 residue types),568,1,568,8920,varies
1,seg_1_PROB,protein (20 residue types),568,569,1136,8920,varies
2,seg_2_ATP,ATP,2,1137,1138,86,43
3,seg_3_MG,MG,2,1139,1140,2,1
4,seg_4_DOPE,DOPE,280,1141,1420,36120,129
5,seg_5_DOPG,DOPG,94,1421,1514,12314,131
6,seg_6_SOD,SOD,218,1515,1732,218,1
7,seg_7_CLA,CLA,108,1733,1840,108,1
8,seg_8_TIP3,TIP3,40512,1841,42352,121536,3
9,seg_9_UNL,UNL,2,42353,42354,172,86


### 4b. ATP atom names (for building selections)

In [ ]:
print("ATP resids:", res["atp_resids"], "| Mg resids:", res["mg_resids"])
print("ATP atom names:", res["atp_atom_names"])

ATP resids: [np.int64(1137), np.int64(1138)] | Mg resids: [np.int64(1139), np.int64(1140)]
ATP atom names: ["C4'", "H4'", "O4'", "C1'", "H1'", 'C5', 'N7', 'C8', 'H8', 'N9', 'N1', 'C2', 'H2', 'N3', 'C4', 'C6', 'N6', 'H61', 'H62', "C2'", "H2''", "O2'", "H2'", "C3'", "H3'", "O3'", 'H3T', "C5'", "H5'", "H5''", "O5'", 'PA', 'O1A', 'O2A', 'O3A', 'PB', 'O1B', 'O2B', 'O3B', 'PG', 'O1G', 'O2G', 'O3G']


### 4c. Walker A and signature-C motifs (sequence scan)

In [ ]:
display(res["motif_table"])
print("Walker A regex: G..G.GK[ST]   |   signature C: LSGGQ (Ser = 2nd residue)")

,segid,walkerA_motif,walkerA_resids,walkerA_Lys,sigC_resids,sigC_Ser
0,seg_0_PROA,GPSGGGKT,365-372,371,470-474,471
1,seg_1_PROB,GPSGGGKT,933-940,939,1038-1042,1039


Walker A regex: G..G.GK[ST]   |   signature C: LSGGQ (Ser = 2nd residue)


### 4d. Composite-site pairing (from starting coordinates)
`closure_CA_CA_start_A` is Walker-A Lys Cα to the partner chain's signature-C Ser Cα, the primary closure coordinate used in the analysis notebook.

In [ ]:
display(res["pairing"])
for _, p in res["pairing"].iterrows():
    print(f"ATP {p.ATP_resid} (Mg {p.paired_Mg_resid}): Walker A of {p.bound_at_WalkerA_of} (Lys{p.WalkerA_Lys}), "
          f"composite partner signature C = {p.composite_SigC_partner} Ser{p.SigC_Ser}; "
          f"start closure {p.closure_CA_CA_start_A} A")

,ATP_resid,bound_at_WalkerA_of,WalkerA_Lys,Lys_NZ_to_PG_A,composite_SigC_partner,SigC_Ser,paired_Mg_resid,Mg_to_ATP_phosphate_A,closure_CA_CA_start_A
0,1137,seg_0_PROA,371,3.67,seg_1_PROB,1039,1139,2.93,25.20
1,1138,seg_1_PROB,939,6.09,seg_0_PROA,471,1140,2.44,28.22


ATP 1137 (Mg 1139): Walker A of seg_0_PROA (Lys371), composite partner signature C = seg_1_PROB Ser1039; start closure 25.2 A
ATP 1138 (Mg 1140): Walker A of seg_1_PROB (Lys939), composite partner signature C = seg_0_PROA Ser471; start closure 28.22 A


### 4e. Q-loop candidates (Gln between Walker A and signature C)
Ranked by distance to the Mg²⁺ of that monomer's site. **Candidate only**: confirm visually (PyMOL/VMD) before relying on it.

In [ ]:
display(res["qloop"])
best = res["qloop"].groupby("segid").head(1)
print("Nearest-to-Mg Gln per chain (Q-loop candidate):")
display(best)

,segid,Gln_resid,dist_to_site_Mg_A
0,seg_0_PROA,413,14.98
1,seg_0_PROA,458,39.51
2,seg_1_PROB,981,10.33
3,seg_1_PROB,1026,40.18


Nearest-to-Mg Gln per chain (Q-loop candidate):


,segid,Gln_resid,dist_to_site_Mg_A
0,seg_0_PROA,413,14.98
2,seg_1_PROB,981,10.33


### 4f. Salt-bridge partner for the chosen Glu
Distances are from Glu OE1/OE2 to every Lys NZ in the system (top 5 per monomer).

In [ ]:
display(res["saltbridge"])

,Glu_segid,Glu_resid,Lys_segid,Lys_resid,dist_A
0,seg_0_PROA,326,seg_0_PROA,323,3.21
1,seg_0_PROA,326,seg_0_PROA,359,23.41
2,seg_0_PROA,326,seg_1_PROB,776,24.18
3,seg_0_PROA,326,seg_0_PROA,100,26.37
4,seg_0_PROA,326,seg_0_PROA,376,28.08
5,seg_1_PROB,894,seg_1_PROB,891,5.07
6,seg_1_PROB,894,seg_1_PROB,927,22.10
7,seg_1_PROB,894,seg_1_PROB,668,23.50
8,seg_1_PROB,894,seg_0_PROA,208,25.22
9,seg_1_PROB,894,seg_1_PROB,944,30.31


### 5. Save the tables

In [ ]:
for name in ["composition", "motif_table", "pairing", "qloop", "saltbridge"]:
    res[name].to_csv(f"{name}.csv", index=False)
files.download("pairing.csv")   # change/add downloads as needed
print("Saved CSVs:", [f"{n}.csv" for n in ["composition","motif_table","pairing","qloop","saltbridge"]])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved CSVs: ['composition.csv', 'motif_table.csv', 'pairing.csv', 'qloop.csv', 'saltbridge.csv']


### Notes
- **Ligand identity:** `UNL` is a generic residue name. The notebook can't tell you it is tariquidar; confirm from your ligand topology.
- **Walker A / signature C** are matched by sequence, so an unusual variant motif may be missed (the table would then be empty).
- **Q-loop** is a proximity-based candidate, not a confirmed assignment.
- **Salt bridge:** if the residue you expect isn't a Lys/Glu pair at ~3-5 Å, the numbering in your source paper may differ from this topology; use the spatial partner instead.